# 02. EDA 및 행동 변수 생성

무료체험 고객의 방문일수·체류시간·입실빈도·첫 방문 시점 등을 생성하고
결제 전환과의 관계를 확인합니다.

**입력:** `master_base.csv`, 정제 방문/로그 데이터  
**출력:** `data/processed/master_model_ready.csv`

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    if current.name == "notebooks":
        return current.parent
    if (current / "notebooks").exists():
        return current
    return current.parent

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SAMPLE_DIR = PROJECT_ROOT / "data" / "sample"
MODEL_DIR = PROJECT_ROOT / "models"

for directory in [INTERIM_DIR, PROCESSED_DIR, SAMPLE_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

master = pd.read_csv(PROCESSED_DIR / "master_base.csv", parse_dates=["trial_date", "first_visit_date"])
visit = pd.read_csv(INTERIM_DIR / "visit_clean.csv", parse_dates=["date"])
log = pd.read_csv(INTERIM_DIR / "access_log_clean.csv", parse_dates=["cdate", "cdate_kst"])
register = pd.read_csv(INTERIM_DIR / "register_clean.csv", parse_dates=["trial_date"])

# 타임존 경계 등으로 4일 방문으로 잡힌 75건 제외
master = master.loc[master["visit_days"] <= 3].copy()

print("체험기간 1~3일 방문자:", len(master))

In [ ]:
# 엔데믹 이후 여부
COVID_END = pd.Timestamp("2023-05-05")
master["is_post_covid"] = (master["trial_date"] > COVID_END).astype(int)

# 체류시간
master["stay_hour"] = master["total_stay_time"] / 3600
master["avg_stay_hour"] = master["stay_hour"] / master["visit_days"]

# 체험기간(신청일 포함 3일) 내 최대 연속 방문일
trial_window = (
    visit
    .merge(register[["user_uuid", "trial_date"]], on="user_uuid", how="inner")
)
trial_window["visit_date"] = trial_window["date"].dt.normalize()
trial_window = trial_window[
    trial_window["visit_date"].between(
        trial_window["trial_date"],
        trial_window["trial_date"] + pd.Timedelta(days=2),
    )
]

def max_consecutive_days(values):
    dates = sorted(pd.Series(values).drop_duplicates())
    if not dates:
        return 0
    best = current = 1
    for i in range(1, len(dates)):
        if (dates[i] - dates[i - 1]).days == 1:
            current += 1
            best = max(best, current)
        else:
            current = 1
    return best

consecutive = (
    trial_window
    .groupby("user_uuid")["visit_date"]
    .apply(max_consecutive_days)
    .reset_index(name="max_consecutive_days")
)

master = master.merge(consecutive, on="user_uuid", how="left")
master["max_consecutive_days"] = master["max_consecutive_days"].fillna(0).astype(int)
master["consecutive_group_2일"] = (master["max_consecutive_days"] >= 2).astype(int)
master["consecutive_group_3일"] = (master["max_consecutive_days"] >= 3).astype(int)

In [ ]:
# 하루 평균 입실 횟수
enter_log = log.loc[log["checkin"] == 1].copy()

daily_enter = (
    enter_log
    .groupby(["user_uuid", enter_log["cdate_kst"].dt.date])
    .size()
    .reset_index(name="daily_enter_count")
)

avg_daily_enter = (
    daily_enter
    .groupby("user_uuid")["daily_enter_count"]
    .mean()
    .reset_index(name="avg_daily_enter")
)

# 첫 방문 시각
first_log = (
    log
    .sort_values(["user_uuid", "cdate_kst"])
    .groupby("user_uuid", as_index=False)
    .first()
)
first_log["first_visit_hour"] = first_log["cdate_kst"].dt.hour

# 방문 지점 수
n_sites = (
    log
    .groupby("user_uuid")["site_id"]
    .nunique()
    .reset_index(name="n_sites_visited")
)

master = (
    master
    .merge(avg_daily_enter, on="user_uuid", how="left")
    .merge(first_log[["user_uuid", "first_visit_hour"]], on="user_uuid", how="left")
    .merge(n_sites, on="user_uuid", how="left")
)

print("avg_daily_enter 결측:", int(master["avg_daily_enter"].isna().sum()))

## 모델링 표본

입실 로그가 없는 고객은 `avg_daily_enter`를 정의할 수 없으므로 제외합니다.
이 과정을 거친 최종 모델링 표본은 **5,629명**입니다.

In [ ]:
required_cols = [
    "avg_stay_hour", "avg_daily_enter", "visit_days", "first_visit_delay",
    "consecutive_group_2일", "consecutive_group_3일",
    "first_visit_hour", "n_sites_visited", "area_pyeong",
    "is_post_covid", "is_payment",
]

model_ready = master.dropna(subset=required_cols).copy()

print("최종 모델링 표본:", len(model_ready))
print("결제율:", round(model_ready["is_payment"].mean(), 4))

assert len(model_ready) == 5629, (
    f"예상 표본 5,629명과 다릅니다. 현재 {len(model_ready):,}명입니다. "
    "원본 데이터 버전 또는 전처리 조건을 확인하세요."
)

In [ ]:
# 주요 EDA 요약
visit_summary = (
    model_ready
    .groupby("visit_days")["is_payment"]
    .agg(["count", "mean"])
    .rename(columns={"count": "users", "mean": "payment_rate"})
)
visit_summary["payment_rate"] = (visit_summary["payment_rate"] * 100).round(2)

covid_summary = (
    model_ready
    .groupby("is_post_covid")["is_payment"]
    .agg(["count", "mean"])
    .rename(columns={"count": "users", "mean": "payment_rate"})
)
covid_summary["payment_rate"] = (covid_summary["payment_rate"] * 100).round(2)

display(visit_summary)
display(covid_summary)

In [ ]:
model_ready.to_csv(
    PROCESSED_DIR / "master_model_ready.csv",
    index=False,
    encoding="utf-8-sig",
)
print("저장 완료: data/processed/master_model_ready.csv")